# Hospitality Forecasting System: Portfolio Walkthrough

## Project Summary
This project develops an end-to-end forecasting pipeline for hospitality operations using anonymised operational data.

The system:
- preprocesses raw rota-style exports
- builds daily sales and labour datasets
- engineers time-series, holiday, payday, and weather features
- benchmarks human forecasts and classical baselines
- trains SARIMAX and XGBoost models
- evaluates forecasting accuracy and business implications

The final objective is not just predictive accuracy, but improving operational decision-making in a hospitality setting.

## 1. Business Problem

Hospitality operations rely heavily on daily demand forecasting for:
- staffing
- labour planning
- wage control
- operational efficiency

In this project, I investigated whether machine learning and time-series methods could improve on manual forecasts and reduce downstream inefficiencies.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..")
sales_path = PROJECT_ROOT / "data" / "processed" / "sales" / "daily_sales_totals_master.csv"
labour_path = PROJECT_ROOT / "data" / "processed" / "labour" / "daily_labour_totals_master.csv"

sales = pd.read_csv(sales_path)
labour = pd.read_csv(labour_path)

sales.head()

## 2. Processed Data

The project uses daily aggregated data derived from anonymised weekly operational files.

Key fields include:
- total realised sales
- manual forecasted sales
- realised labour hours and wages
- forecasted labour hours and wages

In [ ]:
sales["date"] = pd.to_datetime(sales["date"])
labour["date"] = pd.to_datetime(labour["date"])

print("Sales shape:", sales.shape)
print("Labour shape:", labour.shape)
print("Sales date range:", sales["date"].min(), "to", sales["date"].max())

## 3. Feature Engineering

A broad feature set was engineered, including:
- calendar features
- cyclical encodings
- lagged sales features
- bank holiday proximity
- payday effects
- school holiday flags
- weather-derived indicators

A narrower subset was selected for the final XGBoost model.

In [ ]:
features_path = PROJECT_ROOT / "data" / "features" / "engineered_features.csv"
model_features_path = PROJECT_ROOT / "data" / "features" / "model_features.csv"

engineered = pd.read_csv(features_path)
model_df = pd.read_csv(model_features_path)

print("Engineered feature columns:", len(engineered.columns))
print("Final model feature columns:", len(model_df.columns))
model_df.head()

## 4. Baselines

Before training advanced models, simple benchmarks were evaluated:
- naive
- seasonal naive
- rolling 7-day mean
- rolling 14-day mean
- rolling 28-day mean
- weekday average baseline

This establishes whether more advanced models are genuinely adding value.

In [ ]:
baseline_metrics_path = PROJECT_ROOT / "reports" / "results" / "baseline_metrics.csv"
baseline_metrics = pd.read_csv(baseline_metrics_path)
baseline_metrics

## 5. Human Forecast Benchmark

A major strength of this project is that model performance was not only compared to simple statistical baselines, but also to real manual forecasts.

This makes the evaluation more operationally meaningful.

In [ ]:
human_metrics_all = pd.read_csv(PROJECT_ROOT / "reports" / "results" / "human_forecast_metrics_all.csv")
human_metrics_2025 = pd.read_csv(PROJECT_ROOT / "reports" / "results" / "human_forecast_metrics_2025.csv")

human_metrics_all

## 6. SARIMAX Benchmark

A SARIMAX model was trained as a classical time-series benchmark using a small exogenous feature set.

This served as an interpretable statistical reference before the final machine learning model.

In [ ]:
sarimax_metrics = pd.read_csv(PROJECT_ROOT / "reports" / "results" / "sarimax_metrics.csv")
sarimax_metrics

## 7. Final XGBoost Model

The final model used the selected feature subset and was trained using a time-based split.

The selected features combined:
- manual forecast input
- cyclical seasonality
- calendar and holiday context
- payday effects
- weather signal

In [ ]:
xgb_metrics = pd.read_csv(PROJECT_ROOT / "reports" / "results" / "xgboost_metrics.csv")
xgb_metrics

## 8. Visual Forecast Comparison

In [ ]:
from IPython.display import Image

Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "xgboost_forecast_vs_actual.png"))

## 9. Best 8-Week Window

To make results easier to communicate, I also identified the strongest 8-week period where the AI forecast tracked real demand more closely than the manual forecast.

In [ ]:
Image(filename=str(PROJECT_ROOT / "reports" / "figures" / "best_8_week_window.png"))

## 10. Error Analysis

Error analysis was used to identify:
- worst forecast days
- overprediction-heavy periods
- underprediction-heavy periods
- possible regime changes and drift

This was especially important when performance degraded on newer validation periods.

In [ ]:
xgb_predictions = pd.read_csv(PROJECT_ROOT / "reports" / "results" / "xgboost_predictions.csv")
xgb_predictions.head()

## 11. Key Takeaways

### Technical
- Built a modular end-to-end forecasting pipeline
- Compared multiple model families
- Engineered domain-specific time-series features
- Used business-aware evaluation, not just predictive metrics

### Operational
- Machine learning outperformed simple baselines
- Comparison against human forecasts made results decision-relevant
- Validation on later periods highlighted drift and the need for adaptive retraining

### Portfolio value
This project demonstrates not just model training, but full workflow design, evaluation discipline, and business-oriented thinking.